# 01 — The Monge Problem

Ask optimal transport's founding question — *what is the cheapest way to reshape one pile of mass into another?* — and meet Monge's original answer: a deterministic map.

**Prerequisites:** `02/12_bures_distance` (Module 02 complete).

**What you'll be able to do**
- State the optimal-transport problem: a source $\mu$, a target $\nu$, and a ground cost $c(x, y)$.
- Write Monge's formulation as a *deterministic map* $T$ that pushes $\mu$ onto $\nu$.
- Compute the transport cost of a map, and compare two maps to see what "optimal" means.

Welcome to Module 3 — classical optimal transport, the bridge from the information geometry behind you to the quantum transport ahead. It all starts with one homely question: you have a pile of sand shaped like $\mu$ and you want it shaped like $\nu$ — what is the least-effort way to move it? Gaspard Monge asked exactly this in 1781, about hauling earth for fortifications. His answer is where we begin.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.transport.discrete import squared_euclidean_cost

np.random.seed(42)
viz.use_course_style()

## 1. The transport problem, made precise

We hold a **source** distribution $\mu = \sum_i a_i\,\delta_{x_i}$ — mass $a_i$ sitting at position $x_i$ — and we want to reshape it into a **target** $\nu = \sum_j b_j\,\delta_{y_j}$. Both are probability distributions, so $\sum_i a_i = \sum_j b_j = 1$: no mass is created or destroyed, only *moved*.

Moving mass takes effort, and we encode that effort in a **ground cost** $c(x, y) \ge 0$ — the price of carrying one unit from position $x$ to position $y$. The natural choice on the line is the squared distance $c(x, y) = (x - y)^2$, which we use throughout the module. With $\mu$, $\nu$, and $c$ fixed, optimal transport asks a single question:

> *What is the cheapest way to reshape $\mu$ into $\nu$?*

Monge (1781) gave the first answer. It is intuitive — and, as the next notebook shows, sometimes impossible. That tension is the whole story of this arc.

## 2. Monge's formulation — a deterministic map

Monge's idea is to describe the reshaping by a **transport map** $T : X \to Y$ that sends each source position to a single destination. The map has to *deliver the target*: the mass arriving at any target must equal the mass $T$ routed there. This is the **push-forward** condition, written $T_\#\mu = \nu$. Among all maps that satisfy it, Monge seeks the cheapest:

$$ \min_{T :\, T_\#\mu = \nu}\ \int c\big(x, T(x)\big)\,\mathrm{d}\mu(x). $$

The defining feature is that $T$ is a **function**: each source atom sends *all* of its mass to *one* target. That is the picture in your head when you imagine moving a pile — every grain has a single destination. Let's build a case where such a map exists, and see what optimising over maps actually buys us.

In [ ]:
# Equal counts and equal masses on both sides -> a one-to-one (deterministic) map can exist.
x_source = np.array([0.0, 1.0, 2.0])
x_target = np.array([3.0, 5.0, 9.0])
a = np.full(3, 1 / 3)   # uniform source mass
b = np.full(3, 1 / 3)   # uniform target mass

C = squared_euclidean_cost(x_source, x_target)   # C[i, j] = (x_source[i] - x_target[j]) ** 2

# Two candidate Monge maps, each a bijection (source index -> target index).
map_sorted = np.array([0, 1, 2])    # keep the order:  0->3, 1->5, 2->9
map_reversed = np.array([2, 1, 0])  # cross the order: 0->9, 1->5, 2->3

def monge_cost(assignment):
    """Transport cost of a deterministic map: sum_i a_i * c(x_i, y_{T(i)})."""
    return float(sum(a[i] * C[i, assignment[i]] for i in range(len(a))))

print(f"cost of the sorted map   (0->3, 1->5, 2->9): {monge_cost(map_sorted):.3f}")
print(f"cost of the reversed map (0->9, 1->5, 2->3): {monge_cost(map_reversed):.3f}")

**Read the output.** Both assignments are valid Monge maps: each is a function, and each delivers the target exactly (equal masses, one-to-one). Yet they cost different amounts — the **sorted** map, which preserves the order of positions, is the cheaper one. Monge's problem is the *minimisation over all such maps*, and here sorting wins. That "match by sorted rank" is no accident: it is the optimal transport map on the line, and we return to it in `06_1d_quantile_closed_form`.

In [ ]:
# The two piles we must match.
viz.plot_distributions(a, b, source_positions=x_source, target_positions=x_target)
plt.show()

# The sorted map as a plan matrix: P[i, T(i)] = a_i  (all of source i's mass on one target).
plan_sorted = np.zeros_like(C)
for i, j in enumerate(map_sorted):
    plan_sorted[i, j] = a[i]

viz.plot_transport_arrows(
    a, b, plan_sorted,
    source_positions=x_source, target_positions=x_target,
    title="A Monge map: each source sends all its mass to one target",
)
plt.show()

**Read the figures.** The first panel shows the two piles to match: the source (periwinkle) at positions $0, 1, 2$ and the target (amber) at $3, 5, 9$. The second draws the sorted map as flow lines — and the key feature is that **exactly one line leaves each source atom**, carrying *all* of its mass to a single target. That one-arrow-per-source picture is precisely what makes $T$ a function. It is clean, and it is what Monge had in mind.

But it leaned on a quiet assumption: the masses lined up one-to-one. When they do not — when a single source atom must feed *two* different targets — no function can express that. That is exactly where we go next.

## Your turn

1. **Try other matchings.** There are $3! = 6$ bijections between three sources and three targets. Compute `monge_cost` for a couple more (for example `[1, 0, 2]`). Can any of them beat the sorted map?
2. **Change the ground cost.** Replace the squared cost with the absolute distance $c(x, y) = |x - y|$ (build it with `np.abs(x_source[:, None] - x_target[None, :])`). Does the sorted map stay cheapest?
3. **Predict feasibility.** If you raised the first source mass to make `a = [0.5, 0.25, 0.25]` while keeping `b` uniform, could *any* one-to-one map still deliver the target? Reason it through before coding it.

## What you built

- You stated the optimal-transport problem: a source $\mu$, a target $\nu$, and a ground cost $c(x, y)$.
- You wrote Monge's formulation as a deterministic map $T$ obeying the push-forward $T_\#\mu = \nu$.
- You computed the cost of a map and saw that optimising over maps is a genuine choice — the sorted matching came out cheaper.

You have taken the first step of Module 3: you can now say precisely what "moving mass cheaply" means. That one question carries all the way to quantum optimal transport.

## What's next

Monge's map is a lovely picture when it exists. In `02_where_monge_breaks` we build an example where it cannot exist at all — one source atom is forced to split its mass across two targets, and no function permits that. The failure is what motivates Kantorovich's relaxation, the engine of the rest of the module.

## References

- G. Monge, "Mémoire sur la théorie des déblais et des remblais", *Histoire de l'Académie Royale des Sciences de Paris* (1781).
- C. Villani, *Topics in Optimal Transportation*, AMS (2003), ch. 1.
- G. Peyré & M. Cuturi, *Computational Optimal Transport*, NoW (2019), ch. 2. DOI:10.1561/2200000073

**Previous:** `notebooks/02_InformationAndGeometry/12_bures_distance.ipynb` · **Next:** `notebooks/03_ClassicalOptimalTransport/02_where_monge_breaks.ipynb`